In [2]:
from kfp import dsl
from kfp.dsl import (
    Input, Output, Dataset, Model, component, pipeline
)
from google.cloud import aiplatform
from components.components import vertex_hyperparameter_tuning_component,custom_job_training_component,upload_model_component,vertex_batch_predict_bigquery_component
from typing import Any, Dict, List, Optional, Text, Tuple, Union, Sequence, Callable
import yaml
from kfp.v2 import compiler

In [ ]:
! gcloud auth application-default login

In [3]:
config_path = "config.yaml"

with open(config_path, "r") as file:
    config = yaml.safe_load(file)  
    constants = config.get("runtime", {})
    training_cfg = config.get("training", {})

In [4]:
constants

{'project': 'anbc-dev-hcm-cm-de',
 'costcenter': '13070',
 'owner': 'zhongc_aetna_com',
 'tenant': 'hcm-cm-de',
 'pipeline_root': 'gs://hcm-cm-de-code-hcb-dev/vertex-test',
 'location': 'us-east4',
 'service_account': 'gchcb-hcm-cm-de-ontpd@anbc-dev-hcm-cm-de.iam.gserviceaccount.com',
 'cmek_key': 'projects/cvs-key-vault-nonprod/locations/us-east4/keyRings/gkr-nonprod-us-east4/cryptoKeys/gk-anbc-dev-hcm-cm-de-us-east4',
 'labels': {'lob': 'hcb',
  'project': 'anbc-dev-hcm-cm-de',
  'compute_project': 'anbc-dev-hcm-cm-de',
  'shared_project': 'anbc-hcb-dev'}}

#### Define pipeline dag

In [ ]:
@dsl.pipeline(
    name="vertex-hyperparameter-tuning-pipeline",
    description="Pipeline for Vertex AI hyperparameter tuning with containerized component"
)
def hptune_pipeline(
    # Core hyperparameter tuning parameters
    project: str,
    service_account: str,
    cmek_key: str,
    location:str,
    pipeline_root:str,
    model_dir:str,
    # Hyper tune configuration
    hptune_image_uri: str ,
    hptune_machine_type: dict ,
    hptune_package_uris:list,
    hptune_python_module:str,
    hptune_training_args: list,
    parameter_spec_dict: dict,
    eval_metrics: dict,
    max_trials: int,
    parallel_trials: int,
    max_failed_trials: int,
    # Retrain configuration
    training_image_uri: str,
    training_machine_type: dict,
    training_package_uris:list,
    training_python_module:str,
    training_args: list,
    # Register
    model_display_name: str="new_model",
    serving_image_uri: str = "us-docker.pkg.dev/vertex-ai/prediction/xgboost-cpu.1-6:latest",
    model_description: str = None,
    # Pipeline configuration
    upload_to_existing_model: bool=False,
    existing_model_resource_name: str = None,
):


    # Call the hyperparameter tuning component
    hptune_job = vertex_hyperparameter_tuning_component(
        project=project,
        service_account=service_account,
        cmek_key=cmek_key,
        location=location,
        pipeline_root=pipeline_root,
        image_uri=hptune_image_uri,
        package_uris=hptune_package_uris,
        python_module=hptune_python_module,
        training_args=hptune_training_args,
        parameter_spec_dict=parameter_spec_dict,
        eval_metrics=eval_metrics,
        machine_type=hptune_machine_type,
        parallel_trials=parallel_trials,
        max_trials=max_trials,
        max_failed_trials=max_failed_trials,
    )

    retrain_job=custom_job_training_component(project=project,
        service_account=service_account,
        cmek_key=cmek_key,
        location=location,
        image_uri=training_image_uri,
        pipeline_root=pipeline_root,
        package_uris=training_package_uris,
        python_module=training_python_module,
        training_args=training_args,
        machine_type=training_machine_type,
        model_dir=model_dir,
        best_hyperparameters=hptune_job.outputs["best_hyperparameters"]

    )

    model_register=upload_model_component(
        project=project,
        service_account=service_account,
        cmek_key=cmek_key,
        location=location,
        serving_container_image_uri=serving_image_uri,
        model_gcs_location=retrain_job.outputs["model_gcs_location"],
        model_display_name=model_display_name,
        description = model_description,
        upload_to_existing_model=upload_to_existing_model,
        existing_model_resource_name=existing_model_resource_name #f"projects/{constants['project']}/locations/us-east4/models/7725419385405308928"
    )

#### Compile pipeline

In [ ]:
compiler.Compiler().compile(
    pipeline_func=hptune_pipeline,
    package_path="hptune_pipeline_runtime_param_package.yaml"
)
print("Pipeline compiled to hptune_pipeline_runtime_param_package.yaml")

#### Input for the parameter in a dictionary

In [ ]:
pipeline_root=constants["pipeline_root"]
pipeline_name = "vertex-hptune-pipeline-runtime-param-package" 

parameter_values={
    "project": constants["project"],
     "service_account": constants["service_account"],
     "cmek_key": constants["cmek_key"],
     "location": constants["location"],
    "pipeline_root":constants["pipeline_root"],
    "model_dir":f"{pipeline_root}/model",
    "training_package_uris":["gs://hcm-cm-de-data-hcb-dev/mlops-pipeline-test/trainer-0.1.tar.gz"],
    "training_python_module":"trainer.task",
    "training_args":[ "--project", constants["project"],
                      "--training_table", "anbc-hcb-dev.clin_analytics_hcb_dev.mlops_pipeline_test_falls_training",  # Replace with your BigQuery table name
                      "--target_column", "y",  
                      "--model_dir", f"{pipeline_root}/model"  # GCS path to save the model
                 ],
    "hptune_package_uris":["gs://hcm-cm-de-data-hcb-dev/mlops-pipeline-test/trainer-0.1.tar.gz"],
    "hptune_python_module":"trainer.task",
    "hptune_training_args":[ "--project", constants["project"],
                      "--training_table", "anbc-hcb-dev.clin_analytics_hcb_dev.a775382_efm2024_feat_cohort_train_12m_trimmed_feat",  # Replace with your BigQuery table name
                      "--target_column", "y",  
                      "--model_dir", f"{pipeline_root}/model"  # GCS path to save the model
                 ],
    "parameter_spec_dict": {
        "eta": {"type": "double", "min": 0.01, "max": 0.3, "scale": "log"},
        "max_depth": {"type": "integer", "min": 3, "max": 10, "scale": "linear"}
    },
    "eval_metrics": {"roc_auc":"maximize"},  # No serialization
    "max_trials": 10,
    "parallel_trials": 2,
    "max_failed_trials": 2,
    "training_image_uri": "us-docker.pkg.dev/vertex-ai/training/xgboost-cpu.1-6",
    "training_machine_type": {"machine_type": "n1-standard-16",
                        "accelerator_type": None,  # string if not None
                        "accelerator_count": None,  # int if not None
                    },
    "hptune_image_uri": "us-docker.pkg.dev/vertex-ai/training/xgboost-cpu.1-6",
    "hptune_machine_type": {"machine_type": "n1-standard-16",
                        "accelerator_type": None,  # string if not None
                        "accelerator_count": None,  # int if not None
                    },
    "upload_to_existing_model":True,
    "model_display_name":"maternity_model",
    "model_description":"83 features",
    "existing_model_resource_name":""
}
aiplatform.init(project=constants["project"], location=constants["location"],service_account=constants["service_account"])

pipeline_job = aiplatform.PipelineJob(
    display_name=pipeline_name,
    template_path="hptune_pipeline_runtime_param_package.yaml",
    pipeline_root=pipeline_root,
    parameter_values=parameter_values,
    enable_caching=False,
    encryption_spec_key_name=constants["cmek_key"],
)

pipeline_job.run(sync=True)

#### create run with the local compiled yaml template

In [ ]:
aiplatform.init(project=constants["project"], location=constants["location"],service_account=constants["service_account"])

pipeline_job = aiplatform.PipelineJob(
    display_name=pipeline_name,
    template_path="hptune_pipeline_runtime_param_package.yaml",
    pipeline_root=pipeline_root,
    parameter_values=parameter_values,
    enable_caching=False,
    encryption_spec_key_name=constants["cmek_key"],
)

pipeline_job.run(sync=True)

#### Or 1) Upload template to artifect refgistry

In [ ]:
from kfp.registry import RegistryClient
client = RegistryClient(host=f"https://us-east4-kfp.pkg.dev/anbc-dev-hcm-cm-de/hcm-cm-de-kfp-repo")

templateName, versionName = client.upload_pipeline(
  file_name="hptune_pipeline_runtime_param_package.yaml",
  tags=["v3"],
  extra_headers={"description":"This is an hyper tune training pipeline template."})

#### 2) create run on uploaded template with version and tag

In [ ]:
# Replace the argparse cell with this function-based approach:

from components.helper import build_and_upload_training_package, load_config, convert_parameter_spec
# Option 2: Specify custom GCS path
package_uri, version = build_and_upload_training_package(
    gcs_path=f"{constants['pipeline_root']}/packages/trainer-0.3.tar.gz", auto_version=False
)

Building package from training...
Output directory: C:\Users\A974930\Downloads\Github Action Test\vertex-pipeline-demo\src\Training\hyperparameter\dist
STDOUT: running sdist
running egg_info
writing trainer.egg-info\PKG-INFO
writing dependency_links to trainer.egg-info\dependency_links.txt
writing requirements to trainer.egg-info\requires.txt
writing top-level names to trainer.egg-info\top_level.txt
reading manifest file 'trainer.egg-info\SOURCES.txt'
writing manifest file 'trainer.egg-info\SOURCES.txt'
running check
creating trainer-0.1
creating trainer-0.1\trainer
creating trainer-0.1\trainer.egg-info
copying files to trainer-0.1...
copying setup.py -> trainer-0.1
copying trainer\__init__.py -> trainer-0.1\trainer
copying trainer\task.py -> trainer-0.1\trainer
copying trainer.egg-info\PKG-INFO -> trainer-0.1\trainer.egg-info
copying trainer.egg-info\SOURCES.txt -> trainer-0.1\trainer.egg-info
copying trainer.egg-info\dependency_links.txt -> trainer-0.1\trainer.egg-info
copying traine

: 

In [ ]:
aiplatform.init(project=constants["project"], location=constants["location"],service_account=constants["service_account"])

job = aiplatform.PipelineJob(
    display_name="hypertune",
    template_path=f"https://us-east4-kfp.pkg.dev/anbc-dev-hcm-cm-de/hcm-cm-de-kfp-repo/{pipeline_name}/"+"v3",
    parameter_values=parameter_values
)
job.run(sync=True)

### Batch inference pipeline

In [ ]:
@dsl.pipeline(
    name="batch-inference-pipeline",
    description="Pipeline for Vertex AI batch inference with a model in model registry and input and output in BigQuery"
)
def batch_inference_pipeline(
    project: str,
    location: str,
    service_account: str,
    cmek_key: str,
    cost_center: str,
    owner: str,
    #Model details
    model_resource_name: str,
    # BigQuery specific
    key_field: str,  # Unique key field in input table
    input_table: str,  # project.dataset.table
    output_table: str, # project.dataset.table (final output)
    compute_dataset: str, #hcm_cm_de_dec_beam_{ENV}_hcm_cm_de"
    expiration_days: int = 30, # BQ table expiration in days
    # Instance configuration - field filtering
    excluded_fields: list = None,  # Fields to exclude from predictions
    included_fields: list = None,  # Fields to include (if specified, only these will be sent)
    # Machine configuration
    machine_type: dict = {"machine_type": "n2-standard-64"},
    # Job configuration
    starting_replica_count: int = 1,
    max_replica_count: int = 1,
):


    # Call the hyperparameter tuning component
    copy_bq_table = vertex_batch_predict_bigquery_component(
        project=project,
        service_account=service_account,
        cmek_key=cmek_key,
        location=location,
        cost_center= cost_center,
        owner=owner,
        model_resource_name=model_resource_name,
        key_field=key_field,  # Unique key field in input table
        input_table=input_table,  # project.dataset.table
        output_table=output_table, # project.dataset.table (final output)
        compute_dataset=compute_dataset, #hcm_cm_de_dec_beam_{ENV}_hcm_cm_de"
        expiration_days=expiration_days, # BQ table expiration in days
        excluded_fields=excluded_fields,  # Fields to exclude from predictions
        included_fields=included_fields,  # Fields to include (if specified, only these will be sent)
        machine_type=machine_type,
        starting_replica_count=starting_replica_count,
        max_replica_count=max_replica_count,
    )

In [ ]:
compiler.Compiler().compile(
    pipeline_func=batch_inference_pipeline,
    package_path="batch_inference_pipeline.yaml"
)
print("Pipeline compiled to batch_inference_pipeline.yaml")

In [ ]:
parameter_values={
    "project": constants["project"],
     "service_account": constants["service_account"],
     "cmek_key": constants["cmek_key"],
     "location": constants["location"],
     "cost_center":constants["costcenter"],
     "owner":constants["owner"],
    "model_resource_name": "projects/anbc-dev-hcm-cm-de/locations/us-east4/models/2195280517971050496@3",
    # BigQuery specific
    "key_field": "individual_id",  # Unique key field in input table
    "input_table": "anbc-hcb-dev.clin_analytics_hcb_dev.mlops_pipeline_test_falls_training_sample",  # project.dataset.table
    "output_table": "anbc-hcb-dev.clin_analytics_hcb_dev.mlops_test_falls_prediction_v3", # project.dataset.table (final output)
    "compute_dataset": "hcm_cm_de_beam_dev_hcm_cm_de", #hcm_cm_de_dec_beam_{ENV}_hcm_cm_de"
    "expiration_days": 30, # BQ table expiration in days
    # Instance configuration - field filtering
    "excluded_fields":["individual_id","y"] ,  # Fields to exclude from predictions
    # Machine configuration
    "machine_type": {"machine_type": "n1-standard-16"},
    # Job configuration
    "starting_replica_count": 1,
    "max_replica_count": 1,
}
aiplatform.init(project=constants["project"], location=constants["location"],service_account=constants["service_account"])

pipeline_job = aiplatform.PipelineJob(
    display_name="batch_inference_pipeline" ,
    template_path="batch_inference_pipeline.yaml",
    pipeline_root=constants["pipeline_root"],
    parameter_values=parameter_values,
    # enable_caching=False,
    encryption_spec_key_name=constants["cmek_key"],
)
pipeline_job.run(sync=True)